In [ ]:
# Simulazione del funzionamento online per il motor execution (run 5, 9, 13), pugni vs piedi con movimento reale
# Confronta la pipeline CSP cosi' com'e' (offline) con la stessa pipeline vincolata alla
# causalita', cioe' quella che si potrebbe effettivamente eseguire dal vivo, e misura quanto
# costa il vincolo. Le differenze sono due:
#   1. filtro causale IIR invece del filtro a fase zero, che compensa il ritardo guardando
#      i campioni futuri e dal vivo non e' realizzabile
#   2. soglia probabilistica calibrata in cross validation sul training e poi congelata,
#      invece che abbassata guardando quanti campioni del test venivano accettati
# La run di test viene poi riprodotta a blocchi attraverso un buffer circolare, come farebbe
# un sistema reale che riceve i campioni man mano.
# --- Preambolo standard ------------------------------------------------------
# Risale l'albero fino alla root del progetto e rende importabili i moduli in src/util,
# cosi' il notebook funziona indipendentemente dalla cartella da cui parte il kernel.
import sys
from pathlib import Path

PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "dataset_description.json").exists()
)
DATA  = PROJECT_ROOT / "data"
PLOTS = PROJECT_ROOT / "src" / "plots"
sys.path.insert(0, str(PROJECT_ROOT / "src"))
# -----------------------------------------------------------------------------

import time
import warnings
import numpy as np
import mne
from mne.decoding import CSP
from mne_bids import BIDSPath, read_raw_bids
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from sklearn.exceptions import ConvergenceWarning
from util.preprocessing import create_sliding_windows, create_window_labels
from util.online_simulation import (
    CausalBandpass, RingBuffer, average_reference, calibrate_threshold, as_raw
)

warnings.filterwarnings("ignore", category=ConvergenceWarning)
mne.set_log_level('WARNING')

# --- Configurazione ----------------------------------------------------------

root = DATA
runs = ["5", "9", "13"]                       # Le run che vengono prese in considerazione
test_run_order = [runs[2], runs[1], runs[0]]  # Ordine in cui vengono usate le run come test set
first_person = 1
people = 10

window_size = 2         # Lunghezza finestra in secondi
step_size = 0.5         # Lunghezza passo in secondi
label_threshold = 0.8   # Frazione minima di campioni di una classe perche' la finestra le venga assegnata

L_FREQ = 8
H_FREQ = 30
FILTER_ORDER = 4        # Ordine del Butterworth causale usato nella configurazione online

CHANNELS = ["C3", "C4", "Cz", "Fc3", "Fc4", "Fcz", "Cp3", "Cp4", "Cpz"]

# Soglia probabilistica. Offline puo' scendere finche' non accetta MIN_ACCEPTED_RATIO del test;
# online lo stesso criterio viene applicato ai dati di calibrazione e il valore viene congelato.
START_THRESHOLD = 0.90
MIN_THRESHOLD = 0.50
MIN_ACCEPTED_RATIO = 0.70

EVENTS = ["TASK3T0", "TASK3T1", "TASK3T2"]   # rest, classe 1, classe 2

param_grid = {
    "csp__n_components": [2, 4, 6],
    "csp__log": [True],
    "svm__C": [0.1, 1, 10, 100],
    "svm__gamma": ["scale"],
    "svm__kernel": ["rbf", "linear"]
}

def make_pipe():
    return Pipeline([
        ("csp", CSP(reg='ledoit_wolf')),
        ("scaler", StandardScaler()),
        ("svm", SVC(probability=True, class_weight='balanced'))
    ])


In [ ]:
# --- Funzioni di supporto ----------------------------------------------------

def load_run(subject, run):
    bids_path = BIDSPath(
        subject=subject,
        task="motion",
        run=run,
        datatype="eeg",
        root=root,
    )
    raw = read_raw_bids(bids_path, verbose=False)
    events, event_id = mne.events_from_annotations(raw, verbose=False)
    raw.load_data(verbose=False)
    return raw, events, event_id


# Percorso offline: filtro a fase zero, esattamente come nelle pipeline CSP esistenti.
def process_offline(raw):
    raw = raw.copy()
    raw.filter(l_freq=L_FREQ, h_freq=H_FREQ, verbose=False)
    raw.set_eeg_reference('average', projection=False, verbose=False)
    return raw.get_data()


# Percorso causale: filtro IIR in avanti soltanto e riferimento medio, entrambi realizzabili
# dal vivo.
def process_causal(raw):
    sfreq = raw.info['sfreq']
    data = CausalBandpass(L_FREQ, H_FREQ, sfreq, order=FILTER_ORDER)(raw.get_data())
    return average_reference(data)


# Da segnale pre-processato a matrice di finestre, etichette e gruppi, riusando le stesse
# funzioni delle altre pipeline.
def build_dataset(data, info, events, event_id, picks_names):
    raw_p = as_raw(data, info)

    windows, window_samples, step_samples, total_samples = create_sliding_windows(
        raw_p, window_size, step_size
    )
    picks = mne.pick_channels(raw_p.ch_names, picks_names)
    windows = windows[:, picks, :]

    event_map = {
        event_id[EVENTS[0]]: 1,
        event_id[EVENTS[1]]: 2,
        event_id[EVENTS[2]]: 3,
    }
    y, groups = create_window_labels(
        events, event_map, total_samples, window_samples, step_samples,
        threshold=label_threshold, return_groups=True
    )

    mask_active = y != 1
    return windows[mask_active], np.where(y[mask_active] == 2, 0, 1), groups[mask_active], mask_active


# Riproduce la run di test a blocchi di step_samples campioni: filtro causale con stato,
# riferimento medio e buffer circolare. Restituisce le probabilita' emesse,
# una per finestra, nello stesso ordine della sliding window offline.
def replay_online(raw, picks_names, model):
    sfreq = raw.info['sfreq']
    window_samples = int(window_size * sfreq)
    step_samples = int(step_size * sfreq)

    data = raw.get_data()   # segnale grezzo: il pre-processing avviene dentro il ciclo
    bandpass = CausalBandpass(L_FREQ, H_FREQ, sfreq, order=FILTER_ORDER)
    buffer = RingBuffer(data.shape[0], window_samples)
    picks = mne.pick_channels(raw.ch_names, picks_names)

    probs = []

    for k in range(data.shape[1] // step_samples):
        block = data[:, k * step_samples:(k + 1) * step_samples]

        block = bandpass(block)
        block = average_reference(block)
        buffer.push(block)

        if not buffer.ready:
            continue

        window = buffer.window()[picks][np.newaxis, :, :]
        probs.append(model.predict_proba(window)[0])

    return np.array(probs)


# Accuratezza sui soli campioni accettati, insieme alla percentuale di scarti: le due
# grandezze vanno sempre lette in coppia.
def score_with_threshold(probs, y_true, threshold):
    max_probs = np.max(probs, axis=1)
    predictions = np.argmax(probs, axis=1)
    accepted_mask = max_probs >= threshold

    if accepted_mask.sum() < 2 or len(np.unique(y_true[accepted_mask])) < 2:
        return np.nan, 100.0, None

    accuracy = balanced_accuracy_score(y_true[accepted_mask], predictions[accepted_mask])
    discarded = 100 * (1 - accepted_mask.mean())
    cm = confusion_matrix(y_true[accepted_mask], predictions[accepted_mask])
    return accuracy, discarded, cm


In [ ]:
# --- Loop principale ---------------------------------------------------------

results = {"offline": [], "online": []}
discards = {"offline": [], "online": []}
consistency = []   # scarto fra replay a blocchi e stesso pre-processing calcolato in un colpo solo

for i in range(first_person, first_person + people):
    subject = f"{i:03d}"
    subject_start = time.time()

    print("=" * 60)
    print(f"Paziente {subject}")
    print("=" * 60)

    for test_run in test_run_order:
        test_start = time.time()
        train_runs = [run for run in runs if run != test_run]

        cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

        try:
            raw_by_run = {}
            for run in runs:
                raw_by_run[run] = load_run(subject, run)

            # ============ A) Configurazione offline, come le pipeline esistenti ============

            X_tr, y_tr, g_tr = [], [], []
            offset = 0
            for run in train_runs:
                raw, events, event_id = raw_by_run[run]
                data = process_offline(raw)
                Xr, yr, gr, _ = build_dataset(data, raw.info, events, event_id, CHANNELS)
                X_tr.append(Xr); y_tr.append(yr)
                g_tr.append(np.where(gr >= 0, gr + offset, -1))
                offset += len(events)

            X_train = np.concatenate(X_tr); y_train = np.concatenate(y_tr)
            groups_train = np.concatenate(g_tr)

            raw, events, event_id = raw_by_run[test_run]
            data = process_offline(raw)
            X_test, y_test, _, _ = build_dataset(data, raw.info, events, event_id, CHANNELS)

            grid = GridSearchCV(make_pipe(), param_grid, cv=cv,
                                scoring="balanced_accuracy", n_jobs=-1)
            grid.fit(X_train, y_train, groups=groups_train)

            probs_off = grid.predict_proba(X_test)

            # Soglia dinamica scelta guardando il test set, come nelle pipeline offline
            max_probs = np.max(probs_off, axis=1)
            thr_off = START_THRESHOLD
            while np.mean(max_probs >= thr_off) < MIN_ACCEPTED_RATIO and thr_off > MIN_THRESHOLD:
                thr_off -= 0.05

            acc_off, disc_off, _ = score_with_threshold(probs_off, y_test, thr_off)

            # ============ B) Configurazione online, vincolata alla causalita' ============

            # Calibrazione: filtro causale sulle sole run di training
            cal_data = [process_causal(raw_by_run[run][0]) for run in train_runs]

            X_tr, y_tr, g_tr = [], [], []
            offset = 0
            for run, data_c in zip(train_runs, cal_data):
                raw, events, event_id = raw_by_run[run]
                Xr, yr, gr, _ = build_dataset(data_c, raw.info, events, event_id, CHANNELS)
                X_tr.append(Xr); y_tr.append(yr)
                g_tr.append(np.where(gr >= 0, gr + offset, -1))
                offset += len(events)

            X_train_on = np.concatenate(X_tr); y_train_on = np.concatenate(y_tr)
            groups_train_on = np.concatenate(g_tr)

            grid_on = GridSearchCV(make_pipe(), param_grid, cv=cv,
                                   scoring="balanced_accuracy", n_jobs=-1)
            grid_on.fit(X_train_on, y_train_on, groups=groups_train_on)
            model = grid_on.best_estimator_

            # La soglia si calibra sulle probabilita' ottenute in cross validation sul training:
            # dal vivo il test set non esiste ancora.
            probs_cal = cross_val_predict(model, X_train_on, y_train_on,
                                          groups=groups_train_on, cv=cv,
                                          method="predict_proba", n_jobs=-1)
            thr_on = calibrate_threshold(probs_cal, START_THRESHOLD, MIN_THRESHOLD, MIN_ACCEPTED_RATIO)

            # Replay della run di test campione per campione
            raw, events, event_id = raw_by_run[test_run]
            probs_stream = replay_online(raw, CHANNELS, model)

            # Etichette della run di test: identiche a quelle offline, dipendono solo dagli eventi
            data_c = process_causal(raw)
            X_test_on, y_test_on, _, mask_active = build_dataset(
                data_c, raw.info, events, event_id, CHANNELS
            )

            # Allineamento: la finestra i della sliding window offline corrisponde alla i-esima
            # emissione del buffer. Si confronta il prefisso comune.
            n = min(len(probs_stream), len(mask_active))
            probs_on = probs_stream[:n][mask_active[:n]]
            y_on = y_test_on[:probs_on.shape[0]]

            acc_on, disc_on, _ = score_with_threshold(probs_on, y_on, thr_on)

            # Verifica: lo stesso pre-processing calcolato in un colpo solo deve dare le stesse
            # probabilita' del replay a blocchi. Se cosi' non fosse, il buffer sarebbe sbagliato.
            probs_batch = model.predict_proba(X_test_on[:probs_on.shape[0]])
            consistency.append(np.abs(probs_batch - probs_on).max())

            results["offline"].append(acc_off);  discards["offline"].append(disc_off)
            results["online"].append(acc_on);    discards["online"].append(disc_on)

            print(f"  Run test: {test_run}")
            print(f"    offline: {acc_off*100:5.2f}% con {disc_off:4.1f}% di scarti (soglia {thr_off:.2f})")
            print(f"    online : {acc_on*100:5.2f}% con {disc_on:4.1f}% di scarti (soglia {thr_on:.2f})")
            print(f"    delta  : {(acc_on-acc_off)*100:+5.2f} punti | "
                  f"verifica buffer: {consistency[-1]:.2e} | {time.time()-test_start:.1f}s")

        except Exception as e:
            print(f"  Errore {subject} run {test_run}: {e}")

    print(f"  Tempo paziente: {time.time() - subject_start:.1f}s\n")

# --- Riepilogo ---------------------------------------------------------------

off = np.array(results["offline"], dtype=float)
on  = np.array(results["online"],  dtype=float)

sfreq_ref = 160.0
bp = CausalBandpass(L_FREQ, H_FREQ, sfreq_ref, order=FILTER_ORDER)

print("=" * 60)
print(f"Offline: {np.nanmean(off)*100:.2f}% (dev {np.nanstd(off)*100:.2f}) "
      f"con {np.mean(discards['offline']):.1f}% di scarti")
print(f"Online : {np.nanmean(on)*100:.2f}% (dev {np.nanstd(on)*100:.2f}) "
      f"con {np.mean(discards['online']):.1f}% di scarti")
print(f"Costo del vincolo di causalita': {(np.nanmean(on)-np.nanmean(off))*100:+.2f} punti")
print()
print("Verifica del buffer circolare (deve essere ~0):",
      f"{np.max(consistency):.2e}" if consistency else "n/d")
print()
print("Budget di latenza:")
print(f"  eta' del campione piu' vecchio nella finestra: {window_size:.2f} s")
print(f"  intervallo fra due decisioni:                  {step_size:.2f} s")
print(f"  ritardo di gruppo del filtro a 11 Hz:          {bp.group_delay_s(11):.3f} s")
print(f"  ritardo di gruppo del filtro a 20 Hz:          {bp.group_delay_s(20):.3f} s")
